# Single Sample Gene Set Enrichment Analysis (ssGSEA)


Single-sample Gene Set Enrichment Analysis (ssGSEA) is an variation of the GSEA algorithm that instead of calculating enrichment scores for groups of samples (i.e Control vs Disease) and sets of genes (i.e pathways), it provides a score for each each sample and gene set pair (https://www.genepattern.org/modules/docs/ssGSEAProjection/4).


In CKG this function is implemented in the function: **run_ssgsea** (analytics_core.analytics).

To visualize the results, they can be coupled with a Principal Component Analysis (PCA).


In this case, we are applied ssGSEA on two existing projects in CKG: PP4-interactomics experiments in two ovarian cancer cell lines (OVCAR-5 and COV318) expressing N-terminal FLAG- and C-terminal V5-tagged CT45, respectively from Coscia et al 2018 (https://pubmed.ncbi.nlm.nih.gov/30241606/).


Other References: 
- https://www.gsea-msigdb.org/gsea/doc/GSEAUserGuideTEXT.htm

- https://pubmed.ncbi.nlm.nih.gov/16199517/



In [ ]:
from report_manager import project
from analytics_core.analytics import analytics
from analytics_core.viz import viz

from plotly.offline import init_notebook_mode, iplot
%matplotlib inline
init_notebook_mode(connected=True)

### Reading the Project Data

These projects are stored in CKG's graph database and the project reports need to be generated.

In [ ]:
project1 = "P0000004"
project2 = "P0000005"

In [ ]:
p1 = project.Project(project1, datasets={}, knowledge=None, report={}, configuration_files=None)
p1.build_project(force=False)
p1.generate_report()

In [ ]:
p2 = project.Project(project2, datasets={}, knowledge=None, report={}, configuration_files=None)
p2.build_project(force=False)
p2.generate_report()

In [ ]:
proteomics_dataset1 = p1.get_dataset("interactomics")
proteomics_dataset2 = p2.get_dataset("interactomics")

### Gene Ontology Annotation

We can use the Gene Ontology annotation generated during the report creation and available as a dataframe to provide the gene/protein sets for the ssGSEA analysis.

One could also obtain this annotation directly from the graph database using one of the existing Cypher queries and the list of proteins (set).

In [ ]:
annotations1 = proteomics_dataset1.get_dataframe("go annotation")
annotations2 = proteomics_dataset2.get_dataframe("go annotation")

In [ ]:
annotations1

In [ ]:
processed_dataset1 = proteomics_dataset1.get_dataframe('processed')
processed_dataset2 = proteomics_dataset2.get_dataframe('processed')

In [ ]:
processed_dataset1.head()

### ssGSEA

The function needs only the processed dataset (interactomics data matrix) and the annotation matrix. The result provides two dataframes:

- es – Enrichment scores – degree of overrepresentation
- nes – Normalized Enrichment scores – corrects by set size and correlations between gene sets and the dataset

In [ ]:
ssgsea_result1 = analytics.run_ssgsea(processed_dataset1, annotations1, annotation_col='annotation', identifier_col='identifier', set_index=['group', 'sample','subject'], outdir=None, min_size=10, scale=False, permutations=0)

In [ ]:
ssgsea_result1['nes']

In [ ]:
pca_result, args = analytics.run_pca(ssgsea_result1['nes'], drop_cols=['sample', 'subject'], group='group')

In [ ]:
args.update({"loadings":15, "title":'PCA plot groups', 'height':600, 'width':700, 'factor':1})
plot = viz.get_pca_plot(pca_result, identifier='pca', args=args)
iplot(plot.figure)

In [ ]:
ssgsea_result2 = analytics.run_ssgsea(processed_dataset2, annotations2, annotation_col='annotation', identifier_col='identifier', set_index=['group', 'sample','subject'], outdir=None, min_size=10, scale=False, permutations=0)

In [ ]:
ssgsea_result2['nes']

In [ ]:
pca_result, args = analytics.run_pca(ssgsea_result2['nes'], drop_cols=['sample', 'subject'], group='group')

In [ ]:
args.update({"loadings":15, "title":'PCA plot groups', 'height':600, 'width':700, 'factor':1})
plot = viz.get_pca_plot(pca_result, identifier='pca', args=args)
iplot(plot.figure)